# CLensPy: Getting Started

One notebook, one section per physical effect. Each section is the single
source for the matching page under `docs/` — pulled in there via
`{literalinclude}` against this file's jupytext-paired `.py` percent
format, tag-delimited below. Run this notebook top to bottom to reproduce
every snippet in the docs.

## Cosmology

In [1]:
import numpy as np
from clenspy.cosmology import (
    comoving_to_theta,
    fiducial_cosmology,
    growth_factor,
    theta_to_comoving,
)

# every other layer takes a cosmology object as input; build one first.
# a fresh instance every call -- no shared, mutable module-level default.
cosmo = fiducial_cosmology(H0=70.0, Om0=0.3)  # flat LambdaCDM
print(cosmo)

z_lens = 0.35
D_c = np.array([0.1, 1.0, 10.0])  # comoving separations [Mpc]
theta = comoving_to_theta(D_c, z_lens, cosmo, unit="arcmin")
print("theta [arcmin] =", theta)
print("round trip     =", theta_to_comoving(theta, z_lens, cosmo, unit="arcmin"))

z = np.array([0.0, 0.35, 1.0, 2.0])
print("D(z) =", growth_factor(z, cosmo))  # normalised, D(0) = 1

FlatLambdaCDM(H0=70.0 km / (Mpc s), Om0=0.3, Tcmb0=0.0 K, Neff=3.04, m_nu=None, Ob0=0.0)
theta [arcmin] = [ 0.3373975   3.37397499 33.73974985]
round trip     = [ 0.1  1.  10. ]
D(z) = [1.         0.834242   0.61180575 0.4214457 ]


## Power spectrum

In [2]:
from clenspy.cosmology import PkGrid

# CAMB-backed linear P(k, z=0); cached to disk after the first call.
# h-free, like the rest of the package: k in 1/Mpc, P(k) in Mpc^3.
pk_grid = PkGrid(cosmo=cosmo, nonlinear=False)
k_camb = pk_grid.k
Pk_camb = pk_grid(k_camb, z=0.0)
print(f"P(k) from CAMB: k in [{k_camb[0]:.1e}, {k_camb[-1]:.1e}] 1/Mpc, "
      f"P in [{Pk_camb.min():.2e}, {Pk_camb.max():.2e}] Mpc^3")

# physical units end to end: k in 1/Mpc, P in Mpc^3, R in Mpc
from clenspy.cosmology import SigmaGrid

sigma_grid = SigmaGrid(k_camb, Pk_camb)
for r in (1.0, 8.0, 20.0):  # Mpc
    print(f"sigma(R={r:5.1f} Mpc) = {sigma_grid.sigma(r):.4f}")

PkGrid loaded cache file (camb): /Users/jesteves/Documents/Dev/github/CLensPy/src/clenspy/data/pk_cache/f26cce84c16c200a7f9f60cd65e9e758.npz
P(k) from CAMB: k in [1.0e-04, 1.0e+01] 1/Mpc, P in [2.37e-01, 7.32e+04] Mpc^3
sigma(R=  1.0 Mpc) = 2.7395
sigma(R=  8.0 Mpc) = 1.0092
sigma(R= 20.0 Mpc) = 0.5288


## Halo mass function

In [3]:
from clenspy.cosmology import TinkerMassFunction

# cosmo -> PkGrid -> SigmaGrid -> dndlnm_grid, all lazily, on first use:
# the shortcut for the (k, pk) chain built by hand above.
hmf = TinkerMassFunction(cosmo=cosmo)  # Delta = 200 (mean matter) by default

M = np.array([1e13, 1e14, 5e14, 1e15])  # Msun
print("M [Msun]           =", M)
print("dn/dlnM [Mpc^-3]   =", hmf.dndlnm(M, z=0.0))

M [Msun]           = [1.e+13 1.e+14 5.e+14 1.e+15]
PkGrid loaded cache file (camb): /Users/jesteves/Documents/Dev/github/CLensPy/src/clenspy/data/pk_cache/f26cce84c16c200a7f9f60cd65e9e758.npz


dn/dlnM [Mpc^-3]   = [2.44745267e-04 2.35982439e-05 2.30147108e-06 5.33981067e-07]


## Halo bias

In [4]:
from clenspy.cosmology import BiasModel

# same physical chain and the same (M, z) grid idea as the mass function
bias_model = BiasModel(cosmo=cosmo)

print("nu(M)   =", bias_model.nu_at_mass(M))
print("b(M)    =", bias_model.bias(M, z=0.0))

# sigma(M,z) = D(z) sigma(M,0): b(M) rises with z at fixed mass, since a
# fixed mass is a rarer peak against a smaller, less-grown sigma.
for z in (0.0, 0.5, 1.0):
    print(f"b(M=1e14, z={z:.1f}) = {bias_model.bias(1e14, z=z):.4f}")

PkGrid loaded cache file (camb): /Users/jesteves/Documents/Dev/github/CLensPy/src/clenspy/data/pk_cache/f26cce84c16c200a7f9f60cd65e9e758.npz
nu(M)   = [1.11168598 1.71766385 2.46837069 2.93659406]
b(M)    = [1.07022722 1.88289752 3.51663659 4.92177604]
b(M=1e14, z=0.0) = 1.8829
b(M=1e14, z=0.5) = 2.8982
b(M=1e14, z=1.0) = 4.5034


## Concentration-mass relations

In [5]:
from clenspy.cosmology import child18, child18_powerlaw, delta_c, duffy08, m_star_hinv, scatter

# these relations were calibrated in h^-1 Msun and on M_200c, not M_200m --
# see the module docstring's two NOTEs before mixing them with the rest of
# clenspy, which is h-free and M_200m.
m200c_hinv = 1e14  # h^-1 Msun
z = 0.3

ms = m_star_hinv(z)  # Child et al.'s own anchor line, this cosmology's M*
print(f"M_star(z={z}) = {ms:.3e} h^-1 Msun,  M/M_star = {m200c_hinv / ms:.1f}")

c18 = child18(m200c_hinv, z, ms)
c19 = child18_powerlaw(m200c_hinv, z)
cd8 = duffy08(m200c_hinv, z, mass_def="200c")
print(f"c_200c: child18 = {c18:.3f}, child18_powerlaw = {c19:.3f}, "
      f"duffy08 = {cd8:.3f}")

# Duffy08's WMAP-5 sigma_8 sits below Child et al.'s, so it under-predicts
# concentration at cluster scales (Child et al. Fig. 12).
print(f"child18 / duffy08 = {c18 / cd8:.3f}  (> 1, as expected)")

print(f"NFW delta_c(c={c18:.3f}) = {delta_c(c18):.1f}, "
      f"scatter sigma_c = {scatter(c18):.3f}")

M_star(z=0.3) = 1.122e+12 h^-1 Msun,  M/M_star = 89.1
c_200c: child18 = 3.853, child18_powerlaw = 3.831, duffy08 = 3.634
child18 / duffy08 = 1.060  (> 1, as expected)
NFW delta_c(c=3.853) = 4854.4, scatter sigma_c = 1.284


## Halo density profiles

In [6]:
from clenspy.halo import EinastoProfile, NfwProfile

# h-free absolute units: mass in Msun, lengths in Mpc, densities in
# Msun/Mpc^3, wavenumbers in 1/Mpc -- no cosmology object needed, only the
# reference density rho_ref that mass_def is measured against (default:
# the comoving mean matter density, giving M_200m).
m200 = 1e14  # Msun
c200 = 5.0
nfw = NfwProfile(m200=m200, c200=c200)
print(f"r200 = {nfw.r200:.4f} Mpc, rs = {nfw.rs:.4f} Mpc, "
      f"rho_s = {nfw.rho_s:.3e} Msun/Mpc^3")

r = np.array([0.1, 0.5, 1.0, 2.0])  # Mpc
print("rho_NFW(r)     [Msun/Mpc^3] =", nfw.density(r))

# same r_s as the NFW halo (r_s = r200/c200), rho_0 solved so the enclosed
# mass at r200 matches m200 -- a fair shape-only comparison at fixed mass
# and scale radius. alpha=0.25 is a typical cluster shape (Retana-Montenegro
# et al. 2012 report alpha ~ 0.16-0.25 for clusters).
alpha = 0.25
rho0_unit = EinastoProfile(alpha=alpha, rho_0=1.0, r_s=nfw.rs, tol=1e-4)
rho0 = m200 / rho0_unit.enclosed_mass(nfw.r200)
einasto = EinastoProfile(alpha=alpha, rho_0=rho0, r_s=nfw.rs, tol=1e-4)
print("rho_Einasto(r) [Msun/Mpc^3] =", einasto.density(r))

# fourier() returns rho_tilde(k), the *unnormalized* FT -- units of mass,
# going to M as k -> 0, not the dimensionless mass-normalized u(k|M).
k = np.array([0.1, 1.0, 10.0])  # 1/Mpc
print("rho_tilde_NFW(k)     [Msun] =", nfw.fourier(k))
print("rho_tilde_Einasto(k) [Msun] =", einasto.fourier(k))

r200 = 1.4303 Mpc, rs = 0.2861 Mpc, rho_s = 3.547e+14 Msun/Mpc^3
rho_NFW(r)     [Msun/Mpc^3] = [5.57109391e+14 2.68756655e+13 5.02012592e+12 7.94381981e+11]


rho_Einasto(r) [Msun/Mpc^3] = [5.83600877e+14 2.77183496e+13 4.86300049e+12 6.13799586e+11]
rho_tilde_NFW(k)     [Msun] = [9.98998564e+13 9.05346969e+13 8.91993930e+12]


rho_tilde_Einasto(k) [Msun] = [1.83204136e+14 1.06342641e+14 9.27420738e+12]


## Projected density profiles

In [7]:
# same nfw/einasto halos as above; the line-of-sight projection of rho(r).
R = np.array([0.1, 0.5, 1.0, 2.0])  # Mpc, projected radius
print("Sigma_NFW(R)          [Msun/Mpc^2] =", nfw.sigma(R))
print("Sigma_Einasto(R)      [Msun/Mpc^2] =", einasto.sigma(R))
print("DeltaSigma_NFW(R)     [Msun/Mpc^2] =", nfw.deltasigma(R))
print("DeltaSigma_Einasto(R) [Msun/Mpc^2] =", einasto.deltasigma(R))

# both profiles satisfy Sigmabar(<R) = Sigma(R) + DeltaSigma(R) identically,
# even though mean_sigma is evaluated from its own closed form, not this sum
nfw_check = nfw.mean_sigma(R) / (nfw.sigma(R) + nfw.deltasigma(R)) - 1.0
print("NFW Sigmabar consistency, max|rel| =", np.max(np.abs(nfw_check)))

Sigma_NFW(R)          [Msun/Mpc^2] = [1.91291224e+14 3.25021969e+13 1.11712030e+13 3.36400736e+12]
Sigma_Einasto(R)      [Msun/Mpc^2] = [1.97446111e+14 3.17987731e+13 9.66245761e+12 2.13438607e+12]
DeltaSigma_NFW(R)     [Msun/Mpc^2] = [8.51802198e+13 3.87203748e+13 2.00729059e+13 8.74034644e+12]
DeltaSigma_Einasto(R) [Msun/Mpc^2] = [8.89031956e+13 4.03108101e+13 2.08324780e+13 8.73234570e+12]
NFW Sigmabar consistency, max|rel| = 4.218847493575595e-15


## The two-halo term

In [8]:
from clenspy.cosmology import mean_matter_density
from clenspy.halo import TwoHaloTerm

# same (k_camb, Pk_camb) from the power-spectrum section above -- h-free,
# so no k_h/pk_h3 conversion needed here, unlike SigmaGrid/TinkerMassFunction.
z_halo = 0.3
Pk_z = pk_grid(k_camb, z=z_halo)
two_halo = TwoHaloTerm(k_camb, Pk_z, zvec=z_halo)

R = np.array([0.5, 1.0, 5.0, 10.0, 50.0])  # Mpc
xi = two_halo.xi(R, z_halo)
sigma_hat = two_halo.sigma(R, z_halo)
deltasigma_hat = two_halo.deltasigma(R, z_halo)
print("xi(R, z)              =", xi)

# sigma/deltasigma are UNNORMALISED -- units of Mpc, not Msun/Mpc^2 -- until
# multiplied by the comoving (not physical) mean matter density.
rho_m = mean_matter_density(cosmo)
print("Sigma_2h(R)      [Msun/Mpc^2] =", sigma_hat * rho_m)
print("DeltaSigma_2h(R) [Msun/Mpc^2] =", deltasigma_hat * rho_m)

xi(R, z)              = [8.27792441 5.11979202 1.08296537 0.43509318 0.01749311]
Sigma_2h(R)      [Msun/Mpc^2] = [1.87584746e+12 1.66491610e+12 9.28060137e+11 5.78236430e+11
 6.45419485e+10]
DeltaSigma_2h(R) [Msun/Mpc^2] = [1.12356859e+11 1.47545910e+11 2.42598316e+11 2.48223853e+11
 1.23317442e+11]


## The lensing profile (1-halo + 2-halo)

In [9]:
from clenspy.lensing import LensingProfile

# the constructor only stores; nothing is built (no Boltzmann solver call)
# until the first observable is evaluated -- see the class Notes.
lp = LensingProfile(z_cluster=0.3, m200=1e14, concentration=4.0)
print(lp)

R = np.array([0.1, 0.5, 1.0, 5.0])  # Mpc
ds = lp.deltasigma(R)
print("DeltaSigma [Msun/Mpc^2] =", ds)
print(f"b(M) = {lp.bias:.3f}   Sigma_crit = {lp.sigma_crit:.3e} Msun/Mpc^2")

# the 2-halo term is the correlated large-scale structure around the halo
# (TwoHaloTerm); it only matters at large R
ds_1h = LensingProfile(z_cluster=0.3, m200=1e14, include_2halo=False).deltasigma(R)
print("1-halo only             =", ds_1h)
print("2-halo fraction         =", 1.0 - ds_1h / ds)

print("shear(R)         =", lp.shear(R))
print("reduced_shear(R) =", lp.reduced_shear(R))

LensingProfile(model=nfw, z_cluster=0.300, m200=1.00e+14, c=4.00), include_2halo=True)
PkGrid loaded cache file (camb): /Users/jesteves/Documents/Dev/github/CLensPy/src/clenspy/data/pk_cache/f26cce84c16c200a7f9f60cd65e9e758.npz
DeltaSigma [Msun/Mpc^2] = [6.75048039e+13 3.51205237e+13 1.96628219e+13 3.29723173e+12]
b(M) = 3.195   Sigma_crit = 2.834e+15 Msun/Mpc^2
1-halo only             = [6.74701550e+13 3.47624083e+13 1.91883408e+13 2.52255942e+12]
2-halo fraction         = [0.00051328 0.01019675 0.02413087 0.23494627]
shear(R)         = [0.02381918 0.01239233 0.00693806 0.00116343]
reduced_shear(R) = [0.02541893 0.01256748 0.00698106 0.00116494]


## Miscentering

In [10]:
from clenspy.lensing import MiscenteringProfile

# miscentered observables are read from a packaged lookup table, never
# integrated at call time -- only NFW is tabulated today.
R = np.array([0.1, 0.3, 1.0, 3.0])  # Mpc
for r_mis in (0.0, 0.2, 1.0):  # Mpc, the assumed-to-true center offset
    p = MiscenteringProfile(z_cluster=0.25, m200=2e14, r_mis=r_mis,
                             include_2halo=False)
    ds = p.deltasigma_mis(R)
    print(f"r_mis={r_mis:.1f} Mpc  DeltaSigma_mis [Msun/Mpc^2] =", ds)

r_mis=0.0 Mpc  DeltaSigma_mis [Msun/Mpc^2] = [8.80580049e+13 6.64058864e+13 3.01267694e+13 8.89043936e+12]
r_mis=0.2 Mpc  DeltaSigma_mis [Msun/Mpc^2] = [-1.92243006e+12  3.41501316e+13  2.90062982e+13  8.86603639e+12]
r_mis=1.0 Mpc  DeltaSigma_mis [Msun/Mpc^2] = [-5.30598152e+10 -5.03125804e+11 -1.22216230e+13  8.20500151e+12]


## Boost factor

In [11]:
from clenspy.selection import boost_factor_nfw

# B(R) is dimensionless and > 1: correlated cluster members diluting the
# source catalogue make the *effective* Sigma_crit larger, so the measured
# DeltaSigma must be multiplied up by B(R) to recover the true signal.
R = np.array([0.1, 0.3, 1.0, 3.0, 10.0])  # Mpc
rs = 0.35  # Mpc, an NFW scale radius for M ~ 1e14
for B0 in (0.05, 0.10, 0.20):
    print(f"B0={B0:.2f}  B(R) =", boost_factor_nfw(R, B0, rs))

B0=0.05  B(R) = [1.05491131 1.01995534 1.003816   1.00057211 1.00005803]
B0=0.10  B(R) = [1.10982262 1.03991067 1.00763199 1.00114423 1.00011605]
B0=0.20  B(R) = [1.21964523 1.07982134 1.01526398 1.00228846 1.00023211]


## The selection function

In [12]:
from clenspy.selection import EmgParams, LogNormalMor, SelectionFunction

# S_ij(M, z) = S_i(M, z) * S_j(z): probability a halo of mass M at z lands
# in richness bin i and redshift bin j. Factorises exactly (see the module
# docstring for why) into a richness piece (Gauss-Legendre + EMG kernel)
# and a redshift piece (Gaussian CDF difference).
lam_edges = np.array([20.0, 30.0, 45.0, 60.0, 200.0])  # DES Y1 richness bins
z_edges = np.array([0.20, 0.35, 0.50, 0.65])
params = EmgParams(delta_mu=-1.5, sigma=3.0, f_prj=0.3, tau=0.12)
sel = SelectionFunction(lam_edges, z_edges, LogNormalMor(), params, sigma_z=0.01)
print(sel)

print("\nS_i(M, z=0.3): probability of landing in each richness bin")
for m in (1e13, 5e13, 1e14, 3e14, 1e15):
    s = sel.S_i(np.log(m), 0.3)
    print(f"M={m:8.1e} h^-1 Msun  S_i={np.round(s, 4)}  sum={s.sum():.4f}  "
          f"bracket_miss={sel.residual(np.log(m), 0.3):.1e}")

SelectionFunction(4 richness x 3 redshift bins, LogNormalMor(A=76.9, B=1.02, C=0.29, D=0.23), L=8, n_quad=64)

S_i(M, z=0.3): probability of landing in each richness bin
M= 1.0e+13 h^-1 Msun  S_i=[0.0239 0.0086 0.0014 0.0003]  sum=0.0342  bracket_miss=2.6e-04
M= 5.0e+13 h^-1 Msun  S_i=[0.1262 0.0372 0.0058 0.0011]  sum=0.1704  bracket_miss=3.0e-05
M= 1.0e+14 h^-1 Msun  S_i=[0.4134 0.2507 0.0448 0.0087]  sum=0.7177  bracket_miss=1.0e-05
M= 3.0e+14 h^-1 Msun  S_i=[6.000e-04 2.950e-02 1.686e-01 8.012e-01]  sum=0.9999  bracket_miss=3.1e-06
M= 1.0e+15 h^-1 Msun  S_i=[0.     0.     0.     0.1618]  sum=0.1618  bracket_miss=1.8e-06


## The selection-affected bias b_sel

In [13]:
from clenspy.selection import HodMor, PhysicalMassMor, SelBiasEngine

# a cluster selected at observed richness sits behind extra line-of-sight
# structure, so its effective two-halo bias is not b(M,z) but a
# theta-dependent b_sel interpolating between two plateaus (b_small inside
# the aperture, b_large well outside it).

# analytic stand-ins for hmf/bias/xi_nl, in PHYSICAL Msun -- avoids needing
# CAMB or a sigma grid just to demo the *shape* of b_sel(theta).
def hmf(mass, z):
    m, zz = np.broadcast_arrays(np.asarray(mass, float), np.asarray(z, float))
    return 1e-19 * (m / 1e14) ** -2.0 * np.exp(-m / 5e14) / (1.0 + zz)

def bias_toy(mass, z):
    m, zz = np.broadcast_arrays(np.asarray(mass, float), np.asarray(z, float))
    return 1.0 + 0.9 * (m / 3e14) ** 0.3 * (1.0 + zz) ** 0.5

def xi_nl(r, zob):
    return np.maximum((np.maximum(np.asarray(r, float), 1e-3) / 5.0) ** -1.8, 0.0)

engine = SelBiasEngine(
    cosmology=cosmo, xi_nl=xi_nl, hmf=hmf, bias=bias_toy,
    mor=PhysicalMassMor(HodMor.des_y1(), cosmo.h),
    n_z=32, n_M=16, n_theta=8, n_ltr=40, ltr_grid_size=10,
)
lob, zob = 40.0, 0.4  # observed richness, observed redshift
profile = engine.marginalised_bias(lob, zob)
print(f"theta_lambda = {profile.theta_lambda:.6f} rad, "
      f"b_small = {profile.b_small:.3f}, b_large = {profile.b_large:.3f}")

# b_sel(theta) interpolates smoothly between the two plateaus, 0.5 of the
# way there exactly at theta = theta_lambda by construction
for frac in (0.0, 0.5, 1.0, 2.0, 5.0):
    theta = frac * profile.theta_lambda
    print(f"theta/theta_lambda={frac:4.2f}  b_sel={profile(theta):.4f}")

theta_lambda = 0.001073 rad, b_small = 108.059, b_large = 3.247
theta/theta_lambda=0.00  b_sel=84.7175
theta/theta_lambda=0.50  b_sel=55.6530
theta/theta_lambda=1.00  b_sel=26.5886
theta/theta_lambda=2.00  b_sel=5.6551
theta/theta_lambda=5.00  b_sel=3.2482


## Projection lensing Sigma_prj

In [14]:
from clenspy.cosmology import BiasModel as _BiasModel, TinkerMassFunction as _Tmf
from clenspy.cosmology.pkgrid import PkGrid as _PkGrid
from clenspy.lensing import SigmaPrj, SigmaPrjConfig
from clenspy.selection import XiNL

# the projected two-halo surface density around a richness-selected
# cluster (Costanzi 2026 eq. 13): an exact 2 pi sin(theta) d theta
# angular integral -- no Limber, no Bessel -- of the offset-NFW kernel
# against two channels, rnd (the uniform mean, no b_sel) and cl (the
# correlated excess, carrying b_sel(theta) from the engine above).
# Real halo model this time: PkGrid disk-caches CAMB, so it costs seconds
# once and nothing after.
_tmf = _Tmf(cosmo=cosmo, zvec=np.linspace(0.0, 1.0, 21))
_bm = _BiasModel(cosmo=cosmo, zvec=np.linspace(0.0, 1.0, 21))

xi_real = XiNL(_PkGrid(cosmo=cosmo, nonlinear=True), clip=False)  # signed BAO trough

# default exclusion="counter" (zeroes the neighbour count inside the
# ball): the one mode whose cl channel is the mode-invariant
# random-subtracted excess. Under "ball" the exclusion hole would be
# booked in rnd -- and the default channel="cl" of deltasigma_prj below
# would silently omit it.
prj = SigmaPrj(cosmology=cosmo, xi_nl=xi_real, hmf=_tmf, bias=_bm,
               config=SigmaPrjConfig(los_window="hard",
                                     los_depth=71.4))  # Costanzi-mock window
R_prj = np.array([0.5, 2.0, 8.0, 25.0])  # comoving Mpc
# b_sel from the toy engine above: its SHAPE is right, its amplitude is
# not (see docs/selection_bias.md); the mutually calibrated pipeline is
# validation/validate_sigma_prj_mock.py
sigma_tot = prj.sigma_prj(R_prj, 20.0, 0.5, profile, channel="sum")
parts = prj.components()
print("Sigma_prj(R | lob=20, zob=0.5) [Msun/Mpc^2 comoving]:")
for k, r in enumerate(R_prj):
    print(f"  R={r:5.1f}  rnd={parts['rnd'][k]:.3e}  cl={parts['cl'][k]:.3e}"
          f"  sum={sigma_tot[k]:.3e}")

# DeltaSigma_prj is its OWN integral (the DeltaSigma_mis kernel inside the
# same operator) -- never a reconstruction from Sigma_prj. Its rnd channel
# cancels to a boundary term: the excess functional annihilates constants.
ds = prj.deltasigma_prj(R_prj, 20.0, 0.5, profile)
print("DeltaSigma_prj:", np.array2string(ds, precision=3),
      f"\n  rnd/cl at R=8: {prj.rnd[2] / prj.cl[2]:+.4f} (boundary term only)")

PkGrid loaded cache file (camb): /Users/jesteves/Documents/Dev/github/CLensPy/src/clenspy/data/pk_cache/48dcfbb66ff61a4f71f4f631ae403971.npz
PkGrid loaded cache file (camb): /Users/jesteves/Documents/Dev/github/CLensPy/src/clenspy/data/pk_cache/f26cce84c16c200a7f9f60cd65e9e758.npz


PkGrid loaded cache file (camb): /Users/jesteves/Documents/Dev/github/CLensPy/src/clenspy/data/pk_cache/f26cce84c16c200a7f9f60cd65e9e758.npz
Sigma_prj(R | lob=20, zob=0.5) [Msun/Mpc^2 comoving]:
  R=  0.5  rnd=3.583e+12  cl=3.997e+13  sum=4.355e+13
  R=  2.0  rnd=3.579e+12  cl=2.390e+13  sum=2.748e+13
  R=  8.0  rnd=3.576e+12  cl=2.963e+12  sum=6.539e+12
  R= 25.0  rnd=3.538e+12  cl=8.588e+11  sum=4.397e+12
DeltaSigma_prj: [9.037e+11 8.272e+12 4.705e+12 1.169e+12] 
  rnd/cl at R=8: +0.0001 (boundary term only)


## Survey

In [15]:
from clenspy.survey import Survey, deg2, omega_des_y1, survey_bins

# Survey is the source population only (p(z_s), sigma_gamma, n_src) -- no
# footprint, no bins. Omega(z) and the bin grid are separate, since the
# footprint cancels in the shear but not the counts (see the module NOTE).
survey = Survey.from_config("des_y1")
print(survey)

z_s = np.array([0.3, 0.8, 1.5])
print("p(z_s) =", survey.pz_src(z_s))

bins = survey_bins("des_y1")
print(f"{len(bins)} bins = {bins.n_lam} richness x {bins.n_z} redshift")

z_l = np.array([0.2, 0.35, 0.5, 0.65])
print("Omega(z) [deg^2] =", deg2(omega_des_y1(z_l)))

Survey('DES Y1', sigma_gamma=0.3, n_src_arcmin=6.28, zs=[0, 3])
p(z_s) = [0.65542688 1.15967077 0.06176611]
12 bins = 4 richness x 3 redshift
Omega(z) [deg^2] = [1494.02461986 1502.86763277 1511.32308133  649.07197582]


## The lensing kernel

In [16]:
from clenspy.kernels import LensingKernel, sigma_critical

# Sigma_crit for one lens-source pair: diverges as z_s -> z_l, flattens for
# distant sources.
z_l = 0.35
for z_s in (0.6, 1.0, 1.5, 2.0):
    print(f"z_s={z_s:.2f}  Sigma_crit={sigma_critical(z_l, z_s, cosmo):.3e} "
          "Msun/Mpc^2")

# a real survey's source population averages Sigma_crit^-1 over p(z_s) --
# average the inverse, never invert the average (they differ by the source
# weighting, seen in the ratio below).
lk = LensingKernel(survey=Survey.from_config("des_y1"), cosmology=cosmo)
z_l = np.array([0.2, 0.35, 0.5, 0.65])
inv = lk.mean_inverse_sigma_crit(z_l)
mean = lk.mean_sigma_crit(z_l)
print("\n<Sigma_crit^-1>   =", inv)
print("1/<Sigma_crit>    =", 1.0 / mean)
print("ratio (!= 1)      =", inv * mean)
print("f_src_behind(z_l) =", lk.f_src_behind(z_l))

z_s=0.60  Sigma_crit=4.333e+15 Msun/Mpc^2
z_s=1.00  Sigma_crit=2.796e+15 Msun/Mpc^2
z_s=1.50  Sigma_crit=2.383e+15 Msun/Mpc^2
z_s=2.00  Sigma_crit=2.222e+15 Msun/Mpc^2

<Sigma_crit^-1>   = [3.61149037e-16 4.32085305e-16 3.81417233e-16 2.75419611e-16]
1/<Sigma_crit>    = [3.31695530e-16 3.92057624e-16 4.27008960e-16 5.01833643e-16]
ratio (!= 1)      = [1.08879682 1.10209642 0.89323005 0.54882652]
f_src_behind(z_l) = [0.96883223 0.87738012 0.72350911 0.53406002]


## Number counts

In [17]:
from clenspy.observables import ClusterCounts
from clenspy.selection import EmgParams, LogNormalMor, SelectionFunction
from clenspy.survey import omega_des_y1

# the counts are one contraction of the weight W_ij = Omega(z) dV/dz
# n(M,z) S_ij(M,z) -- a smooth analytic dn/dlnM stand-in avoids needing
# CAMB/sigma-grid just to demo the contraction.
def toy_mass_function(ln_mass, z):
    lnm, zz = np.broadcast_arrays(np.asarray(ln_mass, float), np.asarray(z, float))
    m = np.exp(lnm)
    return 1e-5 * (m / 1e14) ** -1.0 * np.exp(-m / 5e14) / (1.0 + zz)

sel = SelectionFunction(
    np.array([20.0, 30.0, 45.0, 60.0, 200.0]),
    np.array([0.20, 0.35, 0.50, 0.65]),
    LogNormalMor(), EmgParams(-1.5, 3.0, 0.3, 0.12), sigma_z=0.01,
)
ln_mass_grid = np.log(np.logspace(13.5, 15.3, 24))  # h^-1 Msun
z_grid = np.linspace(0.16, 0.70, 32)
abundance = ClusterCounts(ln_mass_grid, z_grid, toy_mass_function, sel, cosmo,
                          omega_des_y1)

print("<N_ij> (richness x redshift bins):")
print(abundance.counts())

<N_ij> (richness x redshift bins):
[[345.66171541 641.92848156 772.89742931]
 [228.13035905 427.1340156  518.0046128 ]
 [ 97.1349326  183.1411802  223.43978179]
 [129.85585645 247.75454502 305.22110145]]


## Stacked shear

In [18]:
from clenspy.halo import NfwProfile
from clenspy.observables import StackedDeltaSigma

# the second contraction of the SAME weight `abundance` above, now against
# DeltaSigma(R|M,z) instead of 1 -- this is the halo's own (one-halo) profile;
# a real stacked measurement also carries the projected two-halo excess
# Sigma_prj computed in "Projection lensing" above, evaluated at the bin's
# representative (lambda_ob, z_ob) rather than contracted through W_ij.
radii = np.logspace(-1.0, 1.0, 6)  # Mpc

def nfw_deltasigma(r, mass, z_cluster):
    rho_m = cosmo.critical_density0.to_value("Msun/Mpc^3") * cosmo.Om0
    return NfwProfile(m200=mass, c200=4.0, rho_ref=rho_m).deltasigma(r)

stack = StackedDeltaSigma.from_profile(abundance, nfw_deltasigma, radii)
ds = stack.profile()
print("DeltaSigma_ij^1h(R) [Msun/Mpc^2], lowest redshift bin, rises with richness:")
print(ds[:, 0])

# the identity that proves the stack IS the counts' own weight
ones = np.ones_like(stack.profile_grid)
print("\nstacking DeltaSigma=1, max|result - 1| ="
      f" {np.max(np.abs(abundance.average(ones) - 1.0)):.2e}")

DeltaSigma_ij^1h(R) [Msun/Mpc^2], lowest redshift bin, rises with richness:
[[6.97978965e+13 5.38288665e+13 3.09880765e+13 1.26630791e+13
  3.88154204e+12 9.77650143e+11]
 [8.00754870e+13 6.36057002e+13 3.82833052e+13 1.63623742e+13
  5.18726305e+12 1.33535666e+12]
 [9.28192399e+13 7.58913343e+13 4.77750289e+13 2.13872569e+13
  7.02293569e+12 1.84960860e+12]
 [1.17797585e+14 1.00464460e+14 6.79638090e+13 3.30144419e+13
  1.15866261e+13 3.19108647e+12]]

stacking DeltaSigma=1, max|result - 1| = 0.00e+00


## Shear projection

In [19]:
# the total stacked shear a driver actually fits: the one-halo term (a
# representative M_200m for a lambda_ob=20 cluster, same illustrative
# mass/concentration as "The Lensing Profile" above) plus the projected
# two-halo excess from "Projection lensing" above -- two separate models,
# summed by hand, since no single class owns both pieces at the binned level.
ds_prj = prj.deltasigma_prj(R_prj, 20.0, 0.5, profile)
rho_m_z0 = cosmo.critical_density0.to_value("Msun/Mpc^3") * cosmo.Om0
ds_1h = NfwProfile(m200=1.0e14, c200=4.0, rho_ref=rho_m_z0).deltasigma(R_prj)
ds_tot = ds_1h + ds_prj
print("DeltaSigma(R) [Msun/Mpc^2] at (lambda_ob=20, z_ob=0.5):")
print(f"{'R [Mpc]':>9s} {'1h':>12s} {'prj':>12s} {'total':>12s} {'prj frac':>10s}")
for k, r in enumerate(R_prj):
    print(f"{r:9.2f} {ds_1h[k]:12.4e} {ds_prj[k]:12.4e} {ds_tot[k]:12.4e} "
          f"{ds_prj[k] / ds_tot[k]:10.4f}")

DeltaSigma(R) [Msun/Mpc^2] at (lambda_ob=20, z_ob=0.5):
  R [Mpc]           1h          prj        total   prj frac
     0.50   3.4762e+13   9.0368e+11   3.5666e+13     0.0253
     2.00   8.8014e+12   8.2721e+12   1.7073e+13     0.4845
     8.00   1.2389e+12   4.7051e+12   5.9440e+12     0.7916
    25.00   1.9426e+11   1.1688e+12   1.3631e+12     0.8575


## Covariance

In [20]:
from clenspy.cosmology import growth_factor
from clenspy.covariance import CountsCovariance

# CountsCovariance: Poisson (shot noise) + sample variance (a coherent
# window mode shared by every cluster in a redshift slice). A DES-Y1-like
# toy counts/bias table; sigma_W(z) = sigma_R(R_eff) * D(z) (linear only).
# sigma_grid is the same real-CAMB SigmaGrid built in the power-spectrum
# section above -- never a toy/analytic P(k).
counts = np.array([[2500.0, 3100.0, 2700.0],
                   [900.0, 1150.0, 1000.0],
                   [300.0, 380.0, 330.0],
                   [110.0, 140.0, 120.0]])
bias = np.array([[2.1, 2.2, 2.3],
                 [2.6, 2.7, 2.8],
                 [3.2, 3.3, 3.5],
                 [4.3, 4.5, 4.8]])
z_mid = np.array([0.28, 0.43, 0.57])

sigma_w = sigma_grid.sigma(120.0, truncate=False) * growth_factor(z_mid)  # R_eff=120 Mpc/h

cc = CountsCovariance(counts, bias, sigma_w)
diag_p = np.sqrt(np.diag(cc.cov_poisson())) / counts.ravel()
diag_s = np.sqrt(np.diag(cc.cov_sample_variance())) / counts.ravel()
print("fractional error by component (Poisson falls with N; sample")
print("variance does not, since it is a coherent mode shared at fixed z):")
print("Poisson       =", np.round(diag_p, 4))
print("sample_var    =", np.round(diag_s, 4))

fractional error by component (Poisson falls with N; sample
variance does not, since it is a coherent mode shared at fixed z):
Poisson       = [0.02   0.018  0.0192 0.0333 0.0295 0.0316 0.0577 0.0513 0.055  0.0953
 0.0845 0.0913]
sample_var    = [0.1531 0.1485 0.1448 0.1896 0.1823 0.1763 0.2333 0.2228 0.2203 0.3135
 0.3038 0.3022]


## Halo-to-halo covariance

In [21]:
from clenspy.cosmology import BiasModel
from clenspy.covariance import DeltaSigmaHaloToHaloCovariance
from clenspy.halo import TwoHaloTerm

# the Gaussian covariance treats halo+matter fields as Gaussian and gives
# the variance of the MEAN profile; each cluster in the stack also carries
# its OWN DeltaSigma (mass, concentration scatter), and the stack of N_cl
# of them inherits that population's covariance -- a sixth, independent
# term, scaling as 1/N_cl. Same abundance object as the observables
# section, and the same real CAMB P(k) at z_eff.
z_eff = 0.28
Pk_eff = pk_grid(k_camb, z=z_eff)
rho_m0 = mean_matter_density(cosmo)
twohalo = TwoHaloTerm(k_camb, Pk_eff, zvec=z_eff)
bias_model = BiasModel(k_camb, Pk_eff, cosmo=cosmo)

intrinsic = DeltaSigmaHaloToHaloCovariance(abundance, twohalo, bias_model,
                                           rho_m0, z_eff=z_eff)
print(intrinsic)

radii = np.logspace(-0.7, 1.0, 6)  # Mpc
mean_ds = intrinsic.mean_profile(radii, 0, 0)
sigma_intr = np.sqrt(np.diag(intrinsic.cov(radii, 0, 0)))
print("\nrichness bin 0, redshift bin 0:")
print("<DeltaSigma>  =", mean_ds)
print("sigma_intr    =", sigma_intr)
print("fractional    =", sigma_intr / mean_ds)

# it rises with richness, because the mass population is broader
print("\nmean fractional sigma_intr by richness bin (rises with richness):")
for i in range(sel.n_lambda_bins):
    s = np.sqrt(np.diag(intrinsic.cov(radii, i, 0)))
    m = intrinsic.mean_profile(radii, i, 0)
    print(f"  bin {i}: {np.mean(s / m):.4f}   N_cl = {abundance.counts()[i, 0]:.1f}")

DeltaSigmaHaloToHaloCovariance(z_eff=0.28, sigma_lnc=0.16, n_c=8)



richness bin 0, redshift bin 0:
<DeltaSigma>  = [5.86470477e+13 3.98339927e+13 2.12403318e+13 8.95993786e+12
 3.13747865e+12 9.65283946e+11]
sigma_intr    = [7.74554573e+11 5.58427676e+11 3.33269945e+11 1.58058081e+11
 6.06036179e+10 1.96238890e+10]
fractional    = [0.01320705 0.01401887 0.01569043 0.01764053 0.01931603 0.02032965]

mean fractional sigma_intr by richness bin (rises with richness):


  bin 0: 0.0167   N_cl = 345.7
  bin 1: 0.0213   N_cl = 228.1


  bin 2: 0.0295   N_cl = 97.1


  bin 3: 0.0268   N_cl = 129.9
